In [ ]:
# ============================================================
# CELL 1: Data Loading & Inspection
# ============================================================
# PURPOSE: Import libraries, mount Google Drive, load the raw
# Excel file, and inspect its structure.
# RULE: We do NOT modify or clean anything in this cell.
#       We only load and look. This keeps our pipeline auditable.
# ============================================================


# --- SECTION 1A: Import Libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Libraries imported successfully")


# --- SECTION 1B: Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted")


# --- SECTION 1C: Define File Paths ---
# *** UPDATE THESE PATHS for your experiment ***
FILE_PATH = '/content/drive/MyDrive/your_folder/sample_data.xlsx'
OUTPUT_FOLDER = '/content/drive/MyDrive/your_folder/figures'

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"✓ Output folder: {OUTPUT_FOLDER}")

if os.path.exists(FILE_PATH):
    print(f"✓ File found: {FILE_PATH}")
else:
    print(f"✗ FILE NOT FOUND. Check your path: {FILE_PATH}")
    print("  Tip: In the left panel of Colab, click the folder")
    print("  icon, navigate to your file, right-click it, and")
    print("  select 'Copy path' to get the exact string.")


# --- SECTION 1D: Load the Raw File ---
raw_df = pd.read_excel(
    FILE_PATH,
    sheet_name=0,     # Load the first sheet
                      # Replace 0 with sheet name if needed
                      # e.g. sheet_name="Sheet1"
    header=None,      # No headers in Magellan export
    index_col=None
)

print(f"✓ File loaded. Raw shape: {raw_df.shape}")


# --- SECTION 1E: Inspect the Raw Structure ---
print("\n--- First 30 rows of raw file ---")
display(raw_df.head(30))

print("\n--- Last 15 rows of raw file ---")
display(raw_df.tail(15))

print("\n--- Data types of each column ---")
print(raw_df.dtypes)

In [ ]:
# ============================================================
# CELL 2: Parsing the Time Blocks
# ============================================================
# PURPOSE: Loop through the raw DataFrame, find each time
# point block, extract the relevant well data, and store
# it in a clean, fully labeled structure.
# ============================================================

import re

# --- SECTION 2A: Define Plate Layout ---
# *** UPDATE THIS SECTION for your experiment ***
# This is the only section that changes between experiments

ROWS_PER_BLOCK = 9  # 1 timestamp row + 8 well rows (A-H)

# Maps well row letter to its offset from the timestamp row
WELL_OFFSETS = {
    'E': 5,
    'F': 6,
    'G': 7
}

# Defines the samples in each well row as
# (sample_name, [col_indices]) tuples
# col_indices are 0-based (Python convention)
# Col 0 in the DataFrame = Column 1 on the physical plate
SAMPLE_MAP = {
    'E': [
        ('Standard_High',   [0, 1, 2]),
        ('Sample_E1',       [3, 4, 5]),
        ('Sample_E2',       [6, 7, 8]),
        ('Sample_E3',       [9, 10, 11])
    ],
    'F': [
        ('Standard_Low',    [0, 1, 2]),
        ('Sample_F1',       [3, 4, 5]),
        ('Sample_F2',       [6, 7, 8]),
        ('Sample_F3',       [9, 10, 11])
    ],
    'G': [
        ('Blank',           [0, 1, 2]),
        ('Sample_G1',       [3, 4, 5]),
        ('Sample_G2',       [6, 7, 8])
    ]
}

print("✓ Plate layout defined")
print(f"  Total triplicate sets: {sum(len(v) for v in SAMPLE_MAP.values())}")


# --- SECTION 2B: Find All Timestamp Row Indices ---
timestamp_rows = []

for idx, value in enumerate(raw_df.iloc[:, 0]):
    if isinstance(value, str) and re.match(r'^\d+s$', value.strip()):
        time_seconds = int(

In [ ]:
# ============================================================
# CELL 3: Blank Correction + Protein Normalization Data
# ============================================================
# PURPOSE:
#   Part A — Define protein amounts for normalization
#   Part B — Calculate time-matched blank means
#   Part C — Apply blank correction
# ============================================================


# --- SECTION 3A: Define Protein Data ---
# *** UPDATE THIS SECTION for your experiment ***

MW_KDA = 56.6        # molecular weight in kDa
                     # update for your specific protein construct
VOLUME_LOADED_UL = 50  # µL of protein loaded per well

MW_UG_PER_UMOL = MW_KDA * 1000  # converts kDa to µg/µmol for calculations

protein_data = {}

# --- Experimental samples ---
# Format: 'Sample_Name': {'ug_per_ul': value, 'notes': 'description'}
experimental_samples = {
    'Sample_E1': {'ug_per_ul': 0.25,
                  'notes': 'SEC peak 1'},
    'Sample_E2': {'ug_per_ul': 0.25,
                  'notes': 'SEC peak 2'},
    'Sample_E3': {'ug_per_ul': 0.25,
                  'notes': 'SEC peak 3'},
    'Sample_F1': {'ug_per_ul': 0.25,
                  'notes': 'SEC peak 4'},
    'Sample_F2': {'ug_per_ul': 0.25,
                  'notes': 'Sample F2'},
    'Sample_F3': {'ug_per_ul': 0.25,
                  'notes': 'Sample F3'},
    'Sample_G1': {'ug_per_ul': 0.25,
                  'notes': 'Sample G1'},
    'Sample_G2': {'ug_per_ul': 0.25,
                  'notes': 'Sample G2'},
}

# Calculate amounts programmatically
for sample_name, info in experimental_samples.items():
    ug_loaded   = info['ug_per_ul'] * VOLUME_LOADED_UL
    umol_loaded = ug_loaded / MW_UG_PER_UMOL
    nmol_loaded = umol_loaded * 1000

    protein_data[sample_name] = {
        'ug_per_ul':   info['ug_per_ul'],
        'ug_loaded':   ug_loaded,
        'umol_loaded': umol_loaded,
        'nmol_loaded': nmol_loaded,
        'notes':       info['notes']
    }

# --- Standards ---
# Stored separately — not normalized by protein amount
pla2_standards = {
    'Standard_High': {'activity_u_per_ml': 5.0,
                      'notes': 'PLA2 high concentration standard'},
    'Standard_Low':  {'activity_u_per_ml': 0.5,
                      'notes': 'PLA2 low concentration standard'},
}

# --- Verification ---
print("✓ Protein data defined\n")
print(f"{'Sample':<25} {'µg/µL':>8} {'µg loaded':>10} "
      f"{'µmol loaded':>12} {'nmol loaded':>12}")
print("-" * 70)

for name, data in protein_data.items():
    print(f"{name:<25} "
          f"{data['ug_per_ul']:>8.2f} "
          f"{data['ug_loaded']:>10.1f} "
          f"{data['umol_loaded']:>12.6f} "
          f"{data['nmol_loaded']:>12.4f}")

print("\n✓ Standards (not protein-normalized):")
for name, data in pla2_standards.items():
    print(f"  {name}: {data['activity_u_per_ml']} U/mL")


# --- SECTION 3B: Calculate Time-Matched Blank Means ---
blank_df = clean_df[clean_df['sample'] == 'Blank'].copy()

blank_df['blank_mean'] = blank_df[['rep_1', 'rep_2', 'rep_3']].mean(axis=1)

blank_lookup = blank_df.set_index('time_s')['blank_mean']

print("\n✓ Blank means calculated")
print("\nBlank mean fluorescence by time point:")
print(f"  {'Time (s)':>10} {'Time (min)':>12} {'Blank Mean':>12}")
print("  " + "-" * 36)
for time_s, mean_val in blank_lookup.items():
    time_min = time_s / 60
    print(f"  {time_s:>10} {time_min:>12.1f} {mean_val:>12.1f}")


# --- SECTION 3C: Apply Blank Correction ---
blanked_df = clean_df.copy()

for rep_col in ['rep_1', 'rep_2', 'rep_3']:
    blanked_df[rep_col] = (
        clean_df[rep_col] -
        clean_df['time_s'].map(blank_lookup)
    )

print("\n✓ Blank correction applied")


# --- SECTION 3D: Verification ---
# Check 1: Blank should be approximately zero
print("\nPost-correction blank values (should be ~0):")
corrected_blank = blanked_df[blanked_df['sample'] == 'Blank']
display(corrected_blank[['time_s', 'rep_1', 'rep_2', 'rep_3']].head(5))

# Check 2: Flag large negative values
all_rep_values = blanked_df[['rep_1', 'rep_2', 'rep_3']].values.flatten()

large_negatives = [(i, v) for i, v in enumerate(all_rep_values)
                   if v < -500]

if large_negatives:
    print(f"\n⚠ Warning: {len(large_negatives)} values below -500 RFU")
    print("  These may warrant investigation")
else:
    print("\n✓ No large negative values found")

# Check 3: Preview corrected standard and sample values
print("\nCorrected Standard_High preview (first 5 time points):")
display(blanked_df[blanked_df['sample'] == 'Standard_High'][
    ['time_s', 'rep_1', 'rep_2', 'rep_3']
].head(5))

print("\nCorrected Sample_E1 preview (first 5 time points):")
display(blanked_df[blanked_df['sample'] == 'Sample_E1'][
    ['time_s', 'rep_1', 'rep_2', 'rep_3']
].head(5))

print(f"\n✓ blanked_df shape: {blanked_df.shape}")
n_expected = len(timestamp_rows) * sum(len(v) for v in SAMPLE_MAP.values())
print(f"  Expected: {blanked_df.shape[0]} rows × {blanked_df.shape[1]} columns")

In [ ]:
# ============================================================
# CELL 4: Means, Standard Deviations, Normalization & Graphing
# ============================================================
# PURPOSE:
#   Part A — Calculate mean and SD for all time points
#   Part B — Extract the endpoint time point
#   Part C — Normalize by nmol protein loaded
#   Part D — Build the bar graph
# ============================================================


# --- SECTION 4A: Calculate Mean and SD Across Replicates ---
summary_df = blanked_df.copy()

summary_df['mean_rfu'] = summary_df[['rep_1', 'rep_2', 'rep_3']].mean(axis=1)
summary_df['sd_rfu']   = summary_df[['rep_1', 'rep_2', 'rep_3']].std(axis=1)

print("✓ Mean and SD calculated")
print(f"\nSummary DataFrame preview:")
display(summary_df[['time_s', 'time_min', 'sample',
                     'mean_rfu', 'sd_rfu']].head(10))


# --- SECTION 4B: Extract the Endpoint ---
# *** UPDATE THESE for your experiment ***
ENDPOINT_MIN = 90     # endpoint time point in minutes
ENDPOINT_S   = 5400   # endpoint in seconds
                      # must match a time point in your data
                      # check available time points from Cell 2

endpoint_df = summary_df[
    summary_df['time_s'] == ENDPOINT_S
].copy()

print(f"\n✓ Endpoint extracted: {ENDPOINT_MIN} min ({ENDPOINT_S}s)")
print(f"  Rows extracted: {len(endpoint_df)} (expected {sum(len(v) for v in SAMPLE_MAP.values())})")
print(f"\nEndpoint data:")
display(endpoint_df[['sample', 'mean_rfu', 'sd_rfu']])


# --- SECTION 4C: Normalize by nmol Protein Loaded ---
endpoint_df['nmol_loaded']        = None
endpoint_df['mean_rfu_per_nmol']  = None
endpoint_df['sd_rfu_per_nmol']    = None

for idx, row in endpoint_df.iterrows():
    sample = row['sample']

    if sample in protein_data:
        nmol = protein_data[sample]['nmol_loaded']
        endpoint_df.at[idx, 'nmol_loaded']       = nmol
        endpoint_df.at[idx, 'mean_rfu_per_nmol'] = row['mean_rfu'] / nmol
        endpoint_df.at[idx, 'sd_rfu_per_nmol']   = row['sd_rfu']  / nmol

    elif sample in pla2_standards:
        endpoint_df.at[idx, 'nmol_loaded'] = None

print("\n✓ Normalization applied")
print("\nNormalized endpoint data:")
display(endpoint_df[['sample', 'mean_rfu', 'sd_rfu',
                      'nmol_loaded', 'mean_rfu_per_nmol',
                      'sd_rfu_per_nmol']])


# --- SECTION 4D: Build the Bar Graph ---

# ── Prepare data subsets ──
sample_plot_df = endpoint_df[
    endpoint_df['sample'].isin(protein_data.keys())
].sort_values('sample').reset_index(drop=True)

std_plot_df = endpoint_df[
    endpoint_df['sample'].isin(pla2_standards.keys())
].sort_values('sample').reset_index(drop=True)

# ── Define colors ──
# *** UPDATE color_map to match your sample names ***
color_map = {
    'Sample_E1': '#2196F3',   # bright blue
    'Sample_E2': '#1565C0',   # medium blue
    'Sample_E3': '#0D47A1',   # dark blue
    'Sample_F1': '#82B1FF',   # light blue
    'Sample_F2': '#FF9800',   # bright orange
    'Sample_F3': '#E65100',   # dark orange
    'Sample_G1': '#FFB74D',   # light orange
    'Sample_G2': '#BF360C',   # deep orange
    'Standard_High': '#388E3C',  # dark green
    'Standard_Low':  '#A5D6A7',  # light green
}

# ── Define display labels ──
# *** UPDATE label_map to match your sample names ***
label_map = {
    'Sample_E1': 'Sample E1',
    'Sample_E2': 'Sample E2',
    'Sample_E3': 'Sample E3',
    'Sample_F1': 'Sample F1',
    'Sample_F2': 'Sample F2',
    'Sample_F3': 'Sample F3',
    'Sample_G1': 'Sample G1',
    'Sample_G2': 'Sample G2',
}

std_label_map = {
    'Standard_High': 'PLA2\n5 U/mL',
    'Standard_Low':  'PLA2\n0.5 U/mL',
}

# ── Build the figure ──
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(14, 7),
    gridspec_kw={'width_ratios': [3, 1]}
)

# ── Left Panel: Experimental Samples ──
ax1.bar(
    x=range(len(sample_plot_df)),
    height=sample_plot_df['mean_rfu_per_nmol'],
    yerr=sample_plot_df['sd_rfu_per_nmol'],
    color=[color_map[s] for s in sample_plot_df['sample']],
    capsize=5,
    edgecolor='black',
    linewidth=0.8,
    error_kw={
        'elinewidth': 1.5,
        'ecolor': 'black',
        'capthick': 1.5
    },
    zorder=3
)

ax1.axhline(y=0, color='black', linewidth=1.0, linestyle='-', zorder=2)

ax1.set_title(
    f'PLA2 Activity at {ENDPOINT_MIN} min\n'
    f'(Blank Corrected, Normalized by nmol Protein)',
    fontsize=13,
    fontweight='bold',
    pad=15
)
ax1.set_xlabel('Sample', fontsize=11)
ax1.set_ylabel('RFU / nmol protein', fontsize=11)
ax1.set_xticks(range(len(sample_plot_df)))
ax1.set_xticklabels(
    [label_map[s] for s in sample_plot_df['sample']],
    rotation=45,
    ha='right',
    fontsize=9
)
ax1.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
ax1.set_axisbelow(True)

# ── Right Panel: Standards ──
ax2.bar(
    x=range(len(std_plot_df)),
    height=std_plot_df['mean_rfu'],
    yerr=std_plot_df['sd_rfu'],
    color=[color_map[s] for s in std_plot_df['sample']],
    capsize=5,
    edgecolor='black',
    linewidth=0.8,
    error_kw={
        'elinewidth': 1.5,
        'ecolor': 'black',
        'capthick': 1.5
    },
    zorder=3
)

ax2.axhline(y=0, color='black', linewidth=1.0, linestyle='-', zorder=2)
ax2.set_title(
    f'PLA2 Standards at {ENDPOINT_MIN} min\n(Blank Corrected, Raw RFU)',
    fontsize=13,
    fontweight='bold',
    pad=15
)
ax2.set_xlabel('Standard', fontsize=11)
ax2.set_ylabel('RFU', fontsize=11)
ax2.set_xticks(range(len(std_plot_df)))
ax2.set_xticklabels(
    [std_label_map[s] for s in std_plot_df['sample']],
    rotation=45,
    ha='right',
    fontsize=9
)
ax2.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
ax2.set_axisbelow(True)

# ── Save and Show ──
plt.tight_layout()

plt.savefig(
    f'{OUTPUT_FOLDER}/PLA2_activity_{ENDPOINT_MIN}min.png',
    dpi=300,
    bbox_inches='tight'
)
print(f"\n✓ Figure saved as 'PLA2_activity_{ENDPOINT_MIN}min.png'")
plt.show()
print("\n✓ Cell 4 complete")

In [ ]:
# ============================================================
# SECTION 2: REUSABLE GRAPHING FUNCTION + HISTORICAL DATA
# ============================================================
# Cells below this point contain:
# - A reusable plot_pla2_activity() function (Cell 5)
# - Individual experiment datasets (Cell 6+)
# - Each dataset calls the function to generate figures
# ============================================================

In [ ]:
# ============================================================
# CELL 5: Reusable PLA2 Activity Graphing Function
# ============================================================
# PURPOSE: Define a function that accepts any experiment
# dataset in our standard format and produces a
# publication-ready figure automatically.
# Define this ONCE — call it from any cell below.
# ============================================================

from matplotlib.patches import Patch
import os

def plot_pla2_activity(
    experiment_data,
    output_folder,
    show_plot=True,
    group_labels=None,
    show_separator=False,
    legend_labels=None
):
    """
    Generates a publication-ready PLA2 activity bar graph.

    Parameters:
    -----------
    experiment_data : dict
        Standardized experiment dictionary containing:
        - name:         str, experiment identifier
        - endpoint_min: int, time point in minutes
        - samples:      list of (label, mean, sd) tuples
        - colors:       list of hex color strings per sample
        - hatches:      list of hatch pattern strings per sample
        - standards:    list of (label, mean, sd) tuples
        - std_colors:   list of hex color strings per standard
        - std_hatches:  list of hatch pattern strings per standard
    output_folder : str
        Path to folder for saving the figure
    show_plot : bool
        Whether to display plot inline (default True)
    group_labels : list of (str, float) tuples or None
        Optional group labels and their x positions
        e.g. [('-DDM', 1.5), ('+DDM', 5.5)]
    show_separator : bool
        Whether to draw vertical dashed line splitting bar groups
        Default False
    legend_labels : tuple of (str, str) or None
        Optional labels for solid and hatched bars
        e.g. ('- DDM', '+ DDM')

    Returns:
    --------
    fig : matplotlib Figure object
    """

    # --- Unpack metadata ---
    name         = experiment_data['name']
    endpoint_min = experiment_data['endpoint_min']

    # --- Unpack sample data ---
    sample_labels  = [s[0] for s in experiment_data['samples']]
    sample_means   = [s[1] for s in experiment_data['samples']]
    sample_sds     = [s[2] for s in experiment_data['samples']]
    sample_colors  = experiment_data['colors']
    sample_hatches = experiment_data['hatches']

    # --- Unpack standard data ---
    std_labels  = [s[0] for s in experiment_data['standards']]
    std_means   = [s[1] for s in experiment_data['standards']]
    std_sds     = [s[2] for s in experiment_data['standards']]
    std_colors  = experiment_data['std_colors']
    std_hatches = experiment_data['std_hatches']

    # --- Build figure ---
    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(14, 7),
        gridspec_kw={'width_ratios': [3, 1]}
    )

    # --- Left panel: Experimental samples ---
    for i, (label, mean, sd, color, hatch) in enumerate(
        zip(sample_labels, sample_means, sample_sds,
            sample_colors, sample_hatches)):

        ax1.bar(
            x=i,
            height=mean,
            yerr=sd,
            color=color,
            hatch=hatch,
            capsize=5,
            edgecolor='black',
            linewidth=0.8,
            error_kw={
                'elinewidth': 1.5,
                'ecolor': 'black',
                'capthick': 1.5
            },
            zorder=3
        )

    ax1.axhline(y=0, color='black', linewidth=1.0, linestyle='-', zorder=2)

    # --- Optional: vertical separator ---
    if show_separator:
        n_per_group = len(sample_labels) // 2
        ax1.axvline(
            x=n_per_group - 0.5,
            color='gray',
            linewidth=1.0,
            linestyle='--',
            alpha=0.5,
            zorder=2
        )

    # --- Optional: group labels ---
    if group_labels is not None:
        y_top = ax1.get_ylim()[1] * 0.95
        for label_text, x_pos in group_labels:
            ax1.text(
                x_pos, y_top, label_text,
                ha='center',
                fontsize=10,
                fontstyle='italic',
                color='#1565C0'
            )

    # --- Optional: hatch legend ---
    if legend_labels is not None:
        solid_label, hatch_label = legend_labels
        legend_elements = [
            Patch(facecolor='white', edgecolor='black',
                  label=solid_label),
            Patch(facecolor='white', edgecolor='black',
                  hatch='///', label=hatch_label),
        ]
        ax1.legend(
            handles=legend_elements,
            loc='upper right',
            fontsize=9,
            framealpha=0.8
        )

    # --- Left panel formatting ---
    ax1.set_title(
        f'{name}\n'
        f'PLA2 Activity at {endpoint_min} min\n'
        f'(Blank Corrected, Normalized by nmol Protein)',
        fontsize=13,
        fontweight='bold',
        pad=15
    )
    ax1.set_xlabel('Sample', fontsize=11)
    ax1.set_ylabel('RFU / nmol protein', fontsize=11)
    ax1.set_xticks(range(len(sample_labels)))
    ax1.set_xticklabels(
        sample_labels,
        rotation=45,
        ha='right',
        fontsize=9
    )
    ax1.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
    ax1.set_axisbelow(True)

    # --- Right panel: Standards ---
    for i, (label, mean, sd, color, hatch) in enumerate(
        zip(std_labels, std_means, std_sds,
            std_colors, std_hatches)):

        ax2.bar(
            x=i,
            height=mean,
            yerr=sd,
            color=color,
            hatch=hatch,
            capsize=5,
            edgecolor='black',
            linewidth=0.8,
            error_kw={
                'elinewidth': 1.5,
                'ecolor': 'black',
                'capthick': 1.5
            },
            zorder=3
        )

    ax2.axhline(y=0, color='black', linewidth=1.0, linestyle='-', zorder=2)

    # --- Right panel formatting ---
    ax2.set_title(
        f'PLA2 Standards at {endpoint_min} min\n'
        f'(Blank Corrected, Raw RFU)',
        fontsize=13,
        fontweight='bold',
        pad=15
    )
    ax2.set_xlabel('Standard', fontsize=11)
    ax2.set_ylabel('RFU', fontsize=11)
    ax2.set_xticks(range(len(std_labels)))
    ax2.set_xticklabels(
        std_labels,
        rotation=45,
        ha='right',
        fontsize=9
    )
    ax2.yaxis.grid(True, linestyle='--', alpha=0.7, zorder=0)
    ax2.set_axisbelow(True)

    # --- Save and show ---
    plt.tight_layout()

    os.makedirs(output_folder, exist_ok=True)

    save_path = f'{output_folder}/{name}_activity_{endpoint_min}min.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Figure saved: {save_path}")

    if show_plot:
        plt.show()

    return fig


print("✓ plot_pla2_activity() function defined")
print("  Optional features: group_labels, show_separator, legend_labels")